# Composing a Custom Module

A `dspy.Module` lets you wire multiple signatures into a single pipeline.  
Here we build a two-step module: classify the domain of a question, then rewrite it into a cleaner search query.

In [2]:
import numpy
import dspy
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

lm = dspy.LM('ollama_chat/llama3.1:8b', api_base='http://localhost:11434')
dspy.configure(lm=lm)

## Step 1 — Define the Signatures

In [3]:
class DomainClassifier(dspy.Signature):
    """Classify the domain of the question."""

    question: str = dspy.InputField(desc="A user's question")
    domain: Literal["science", "history", "sports"] = dspy.OutputField(desc="The domain the question belongs to")


class QueryRewriter(dspy.Signature):
    """Rewrite the question into a precise search query suited to its domain."""

    question: str = dspy.InputField(desc="The original user question")
    domain: str = dspy.InputField(desc="The domain the question belongs to")
    search_query: str = dspy.OutputField(desc="A concise, search-engine-friendly version of the question")

## Step 2 — Build the Module

- Declare sub-modules in `__init__`
- Wire them together in `forward()`
- The output of step 1 feeds into step 2

In [4]:
class QueryUnderstanding(dspy.Module):
    def __init__(self):
        self.classify = dspy.Predict(DomainClassifier)
        self.rewrite = dspy.ChainOfThought(QueryRewriter)

    def forward(self, question):
        classification = self.classify(question=question)
        rewrite = self.rewrite(question=question, domain=classification.domain)
        return dspy.Prediction(
            domain=classification.domain,
            search_query=rewrite.search_query,
            reasoning=rewrite.reasoning
        )

## Step 3 — Run it

In [5]:
pipeline = QueryUnderstanding()

questions = [
    "who scored the most goals in the world cup",
    "why did the roman empire fall",
    "how does a black hole form",
]

for q in questions:
    result = pipeline(question=q)
    print(f"Q:         {q}")
    print(f"Domain:    {result.domain}")
    print(f"Reasoning: {result.reasoning}")
    print(f"Rewrite:   {result.search_query}")
    print()

Q:         who scored the most goals in the world cup
Domain:    sports
Reasoning: To create a search query, I'll need to identify the key entities in the question: the event (world cup) and the action (scored the most goals). This will help me craft a specific search query that targets the desired information within the sports domain.
Rewrite:   (world cup goals scored most)

Q:         why did the roman empire fall
Domain:    history
Reasoning: The Roman Empire's decline was a complex process influenced by various factors, including internal strife, external pressures, economic troubles, and environmental degradation. A precise search query would require identifying specific events, periods, or causes that contributed to its downfall.
Rewrite:   "Roman Empire decline causes" OR "fall of Roman Empire" OR "Roman Empire collapse factors"

Q:         how does a black hole form
Domain:    science
Reasoning: A black hole forms when a massive star collapses in on itself, causing a massive a

In [6]:
dspy.inspect_history(n=2)





[2026-09-08T09:19:46.663531]

System message:

Your input fields are:
1. `question` (str): A user's question
Your output fields are:
1. `domain` (Literal['science', 'history', 'sports']): The domain the question belongs to
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## domain ## ]]
{domain}        # note: the value you produce must exactly match (no extra characters) one of: science; history; sports

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the domain of the question.


User message:

[[ ## question ## ]]
how does a black hole form

Respond with the corresponding output fields, starting with the field `[[ ## domain ## ]]` (must be formatted as a valid Python Literal['science', 'history', 'sports']), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## domain ## ]]
science

[[ ## completed ## ]]





[2026-09-08T09:19:4